# Fink/LSST — Dipole Analysis per Deep Drilling Field by making sliced cones and time slices

This notebook retrieves DIA alerts from the **LSST Deep Drilling Fields** (DDFs)  
via the Fink LSST API, focusing on **dipole characterisation**.

## Scientific goals

1. **Sky map of dipoles**: spatial distribution of `isDipole=True` alerts in (ra, dec),  
   with arrows indicating the dipole orientation (`r:dipoleAngle`).
2. **Temporal evolution of the dipole fraction**: proportion of alerts flagged as dipoles  
   as a function of time (MJD) and visit, per DDF and per filter band.  
   → Did Rubin AP pipeline improvements reduce the dipole rate over time?
3. **Dipole properties**: flux differences, lengths, chi2, position angles —  
   to understand the morphology and origin (mis-registered template, PSF mismatch, …).

## Data strategy

- For each DDF, perform a **conesearch** (`/api/v1/conesearch`) with all dipole columns  
  plus flux, position, crossmatch, and observing metadata.
- Save one **parquet file per DDF** under `data_DIPOLES_01/` for reproducibility.
- Cutouts and full diaSources (`/api/v1/sources`) are intentionally **not fetched here**  
  — they will be loaded in a dedicated notebook when needed for image inspection.

## Dipole columns fetched

| Column | Description |
|--------|-------------|
| `r:isDipole` | True if the source was classified as a dipole |
| `r:isNegative` | True if this is the negative lobe of a dipole |
| `r:dipoleFitAttempted` | True if the dipole fit was attempted |
| `r:dipoleFluxDiff` | Flux difference between positive and negative lobes (nJy) |
| `r:dipoleFluxDiffErr` | Uncertainty on `dipoleFluxDiff` (nJy) |
| `r:dipoleMeanFlux` | Mean of positive and negative lobe fluxes (nJy) |
| `r:dipoleMeanFluxErr` | Uncertainty on `dipoleMeanFlux` (nJy) |
| `r:dipoleLength` | Angular separation between lobes (arcsec) |
| `r:dipoleAngle` | Position angle of the dipole axis (degrees, N through E) |
| `r:dipoleNdata` | Number of pixels used in the dipole fit |
| `r:dipoleChi2` | Chi² of the dipole fit |

## Flux and observing metadata columns

| Column | Description |
|--------|-------------|
| `r:ra`, `r:dec` | Sky position (degrees) |
| `r:midpointMjdTai` | Observation epoch (MJD TAI) |
| `r:band` | Filter band (u/g/r/i/z/y) |
| `r:diaObjectId` | DIA object identifier |
| `r:diaSourceId` | DIA source (detection) identifier |
| `r:psfFlux`, `r:psfFluxErr` | PSF-fit flux and uncertainty (nJy) |
| `r:scienceFlux`, `r:scienceFluxErr` | Science image flux at source position (nJy) |
| `r:templateFlux`, `r:templateFluxErr` | Template image flux at source position (nJy) |
| `r:apFlux`, `r:apFluxErr` | Aperture flux (nJy) |
| `r:visit` | Visit ID (13-digit, e.g. 2026030900027) |
| `r:detector` | CCD detector number (0–188) |
| `r:x`, `r:y` | Pixel position on CCD |
| `r:extendedness` | Morphology flag (0=point-source, 1=extended) |
| `r:reliability` | Alert reliability score [0–1] |
| `r:nDiaSources` | Number of detections for this diaObject |

## Key crossmatch columns

| Column | Catalogue |
|--------|-----------|
| `f:xm_gaiadr3_DR3Name` | Gaia DR3 source name |
| `f:xm_gaiadr3_VarFlag` | Gaia DR3 variability flag |
| `f:xm_simbad_otype` | SIMBAD object type |
| `f:xm_legacydr8_pstar` | Legacy DR8 P(star) |
| `f:xm_tns_fullname` | TNS transient name |
| `f:clf_cats_class` | Fink ML class |
| `f:clf_cats_score` | Fink ML score |


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-20
- last update : 2026-05-21

## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import time
import warnings
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrowPatch
from astropy.time import Time
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

In [ ]:
# ── API base URL ──────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org"

# ── Cone search parameters ────────────────────────────────────────────────────
CONE_RADIUS = 3600.0  # arcsec — 1.0 degree radius per DDF
# CONE_RADIUS = 1000.0  # arcsec — 1.0 degree radius per DDF
N_MAX = 5000  # max number of alerts per conesearch call

# if time slicing is required in the API
STARTTIME = datetime.strptime("2025-09-06 00:00:00", "%Y-%m-%d %H:%M:%S")
STOPTIME = datetime.strptime("2026-05-31 00:00:00", "%Y-%m-%d %H:%M:%S")
STEPDAYS = 15

print(STARTTIME, STOPTIME, STEPDAYS)

# ── LSST Deep Drilling Fields (RA/Dec J2000) ─────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "DIPOLES_01b"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

# ── Plotting style ────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str):
    """Save figure to both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Column definitions

We define the explicit column list for the conesearch.  
Using an explicit list instead of `None` (all columns) keeps responses compact  
and avoids server-side timeouts on large DDFs.

In [ ]:
# ── Dipole columns ────────────────────────────────────────────────────────────
DIPOLE_COLS = [
    "r:isDipole",
    "r:isNegative",
    "r:dipoleFitAttempted",
    "r:dipoleFluxDiff",
    "r:dipoleFluxDiffErr",
    "r:dipoleMeanFlux",
    "r:dipoleMeanFluxErr",
    "r:dipoleLength",
    "r:dipoleAngle",
    "r:dipoleNdata",
    "r:dipoleChi2",
]

# ── Flux and source columns ───────────────────────────────────────────────────
FLUX_COLS = [
    "r:ra",
    "r:dec",
    "r:midpointMjdTai",
    "r:band",
    "r:diaObjectId",
    "r:diaSourceId",
    "r:psfFlux",
    "r:psfFluxErr",
    "r:scienceFlux",
    "r:scienceFluxErr",
    "r:templateFlux",
    "r:templateFluxErr",
    "r:apFlux",
    "r:apFluxErr",
    "r:nDiaSources",
    "r:extendedness",
    "r:reliability",
    "r:visit",
    "r:detector",
    "r:x",
    "r:y",
]

# ── Crossmatch columns ────────────────────────────────────────────────────────
CROSSMATCH_COLS = [
    "f:xm_gaiadr3_DR3Name",
    "f:xm_gaiadr3_VarFlag",
    "f:xm_gaiadr3_Plx",
    "f:xm_gaiadr3_e_Plx",
    "f:xm_gaiadr3_PhotGMag",
    "f:xm_simbad_otype",
    "f:xm_legacydr8_pstar",
    "f:xm_legacydr8_zphot",
    "f:xm_legacydr8_fqual",
    "f:xm_tns_fullname",
    "f:xm_tns_type",
    "f:xm_vsx_Type",
    "f:xm_gcvs_type",
    "f:xm_mangrove_2MASS_name",
    "f:xm_mangrove_HyperLEDA_name",
    "f:clf_cats_class",
    "f:clf_cats_score",
    "f:clf_snnSnVsOthers_score",
    "f:is_sso",
]

# ── Full column string for the API call ───────────────────────────────────────
ALL_COLS = FLUX_COLS + DIPOLE_COLS + CROSSMATCH_COLS
COLUMNS_STR = ",".join(ALL_COLS)

# Compact payload for date-window requests (avoids 400 on large fields like COSMOS)
COLUMNS_SLIM = ",".join(FLUX_COLS + DIPOLE_COLS)

print(f"Total columns requested: {len(ALL_COLS)}")
print(f"  Flux+position : {len(FLUX_COLS)}")
print(f"  Dipole        : {len(DIPOLE_COLS)}")
print(f"  Crossmatch    : {len(CROSSMATCH_COLS)}")

In [ ]:
print(f"COLUMNS_STR = {COLUMNS_STR}")

## 3. API wrappers

In [ ]:
def _post_json(url: str, payload: dict, timeout: int = 120) -> list | dict:
    """POST JSON and return parsed response; include response body for HTTP errors."""
    r = requests.post(url, json=payload, timeout=timeout)
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        body = (r.text or "").strip()
        msg = f"{e}"
        if body:
            msg += f" | response: {body[:500]}"
        raise requests.HTTPError(msg, response=r) from e
    return r.json()


def fetch_conesearch(
    ra: float,
    dec: float,
    radius: float,
    n: int = N_MAX,
    columns: str | None = COLUMNS_STR,
) -> pd.DataFrame:
    """
    Cone search via /api/v1/conesearch — returns one row per alert (diaSource).

    IMPORTANT: column names must use the 'r:' or 'f:' prefix.
    The 'i:' prefix is NOT supported here and causes HTTP 500 errors.

    Parameters
    ----------
    ra, dec  : field centre (degrees)
    radius   : search radius (arcsec)
    n        : max alerts to return
    columns  : comma-separated column string (None = all columns)

    Returns
    -------
    pd.DataFrame — one row per alert, or empty DataFrame on failure.
    """
    payload = {
        "ra": ra,
        "dec": dec,
        "radius": radius,
        "n": n,
        "output-format": "json",
    }

    print("fetch_conesearch payload : ", payload)

    if columns:
        payload["columns"] = columns
    try:
        raw = _post_json(f"{FINK_API}/api/v1/conesearch", payload)
        if not raw:
            return pd.DataFrame()
        return pd.DataFrame(raw)
    except Exception as e:
        print(f"fetch_conesearch ERROR (ra={ra:.3f}, dec={dec:.3f}): {e}")
        return pd.DataFrame()


def _fetch_timerange_once(
    ra: float,
    dec: float,
    radius: float,
    n: int,
    startdate: str,
    stopdate: str,
    columns: str | None,
) -> pd.DataFrame:
    t0 = datetime.strptime(startdate, "%Y-%m-%d %H:%M:%S")
    t1 = datetime.strptime(stopdate, "%Y-%m-%d %H:%M:%S")
    window_days = max((t1 - t0).total_seconds() / 86400.0, 1e-6)

    payload = {
        "ra": ra,
        "dec": dec,
        "radius": radius,
        "n": int(n),
        "startdate": startdate,
        "window": float(window_days),
        "output-format": "json",
    }

    if columns:
        cols = [c.strip() for c in columns.split(",") if c.strip()]
        if "f:firstDiaSourceMjdTaiFink" not in cols:
            cols.append("f:firstDiaSourceMjdTaiFink")
        payload["columns"] = ",".join(cols)

    print("fetch_conesearch_timerange payload : ", payload)
    raw = _post_json(f"{FINK_API}/api/v1/conesearch", payload, timeout=180)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_conesearch_timerange(
    ra: float,
    dec: float,
    radius: float,
    n: int = N_MAX,
    startdate: str = "2025-09-06 00:00:00",
    stopdate: str = "2026-09-06 00:00:00",
    columns: str
    | None = "r:diaSourceId,r:diaObjectId,r:midpointMjdTai,r:band,r:isDipole,r:dipoleFitAttempted,r:dipoleChi2",
) -> pd.DataFrame:
    """
    Cone search avec fenêtre temporelle, robuste aux 5xx/timeout de COSMOS.
    """
    # progressive degradation for heavy COSMOS windows
    attempts = [
        (radius, n),
        (min(radius, 2400.0), min(n, 2500)),
        (min(radius, 1800.0), min(n, 1200)),
        (min(radius, 1200.0), min(n, 800)),
    ]

    last_err = None
    for radius_i, n_i in attempts:
        try:
            return _fetch_timerange_once(
                ra=ra,
                dec=dec,
                radius=radius_i,
                n=n_i,
                startdate=startdate,
                stopdate=stopdate,
                columns=columns,
            )
        except Exception as e:
            last_err = e
            print(f"[WARN] retry with radius={radius_i:.0f} n={n_i}: {e}")
            time.sleep(1.0)

    print(f"[ERROR] {startdate} → {stopdate} : {last_err}")
    return pd.DataFrame()


def generate_time_slices(start, stop, step_days=3):
    """
    Génère des intervalles [start, stop] de longueur step_days
    """
    current = start
    while current < stop:
        next_ = current + timedelta(days=step_days)
        yield current, min(next_, stop)
        current = next_


def fetch_conesearch_sliced(
    ra: float,
    dec: float,
    radius: float,
    n: int,
    start: str,
    stop: str,
    step_days: int = 15,
    columns: str | None = COLUMNS_STR,
) -> pd.DataFrame:
    # columns: str | None = "r:diaSourceId,r:diaObjectId,r:midpointMjdTai,r:band,r:isDipole,r:dipoleFitAttempted,r:dipoleChi2",
    # ) -> pd.DataFrame:
    """Cone search découpe temporelle + fallback spatial adaptatif couvrant tout le cone."""
    dfs = []

    for t0, t1 in generate_time_slices(start, stop, step_days):
        start_str = t0.strftime("%Y-%m-%d %H:%M:%S")
        stop_str = t1.strftime("%Y-%m-%d %H:%M:%S")

        print(f"→ Fetch {start_str} → {stop_str}")

        # 1) tentative cone unique
        df = fetch_conesearch_timerange(ra, dec, radius, n, start_str, stop_str, columns)
        if not df.empty:
            dfs.append(df)
            continue

        # 2) fallback spatial adaptatif
        tile_radius = min(600.0, max(250.0, radius / 3.5))  # arcsec
        step_arcsec = 0.95 * np.sqrt(2.0) * tile_radius
        nside = int(np.ceil((2.0 * radius) / step_arcsec)) + 1
        if nside % 2 == 0:
            nside += 1
        offs_arcsec = np.linspace(-radius, radius, nside)

        print(f"   fallback tiles: nside={nside}, tile_radius={tile_radius:.0f}, step~{step_arcsec:.0f}")

        tile_dfs = []
        for dx_as in offs_arcsec:
            for dy_as in offs_arcsec:
                # ignorer les centres trop loin du cone principal
                if np.hypot(dx_as, dy_as) > (radius + tile_radius):
                    continue

                ra_i = ra + dx_as / 3600.0
                dec_i = dec + dy_as / 3600.0

                dfi = fetch_conesearch_timerange(
                    ra_i,
                    dec_i,
                    tile_radius,
                    min(n, 500),
                    start_str,
                    stop_str,
                    columns,
                )

                if not dfi.empty:
                    tile_dfs.append(dfi)
                time.sleep(0.15)

        if tile_dfs:
            dft = pd.concat(tile_dfs, ignore_index=True)
            if "r:diaSourceId" in dft.columns:
                dft = dft.drop_duplicates(subset="r:diaSourceId")
            dfs.append(dft)

    if not dfs:
        return pd.DataFrame()

    df_all = pd.concat(dfs, ignore_index=True)
    if "r:diaSourceId" in df_all.columns:
        df_all = df_all.drop_duplicates(subset="r:diaSourceId")
    return df_all


def try_fast_search(ra, dec):
    """ """
    df = fetch_conesearch(
        ra,
        dec,
        radius=10,  # très petit
        n=10000,
        # columns="r:ra,r:dec"
    )
    return df


def try_slicing_search(ra, dec, rmax=1000):
    """ """
    dfs = []
    count = 0
    for dra in np.linspace(-0.5, 0.5, 5):  # degrés !
        for ddec in np.linspace(-0.5, 0.5, 5):
            count += 1
            df = fetch_conesearch(
                ra + dra,
                dec + ddec,
                radius=rmax,
                n=10000,
                # columns="r:ra,r:dec"
            )
            n_slice = len(df)
            print(f" \t - {count}) try_slicing_search at ({dra}, {ddec}) =>  n_slice = {n_slice}")
            dfs.append(df)
    df = pd.concat(dfs).drop_duplicates()
    return df


print("API helpers defined.")

## 4. Fetch alerts per DDF and save to parquet

Each DDF is queried independently.  
Results are saved to `data_DIPOLES_01/<field_name>_alerts.parquet`.

If the parquet file already exists, it is reloaded from disk (no API call) —  
set `FORCE_RELOAD = True` to re-fetch from the API regardless.

In [ ]:
# ── Toggle: set True to ignore cached parquet and re-fetch from API ───────────
FORCE_RELOAD = False

ddf_alerts: dict[str, pd.DataFrame] = {}


# loop on DDF
for field_name, (ra, dec) in DEEP_FIELDS.items():
    parquet_path = os.path.join(DIR_DATA, f"{field_name}_alerts.parquet")
    safe_name = field_name.replace("-", "_").replace(" ", "_")

    if os.path.exists(parquet_path) and not FORCE_RELOAD:
        df = pd.read_parquet(parquet_path)
        print(f"[{field_name:12s}] Loaded from cache: {len(df):6d} alerts")
    else:
        print(f"[{field_name:12s}] Fetching from API  (ra={ra:.4f}, dec={dec:.4f}) …", end=" ")
        t0 = time.time()
        df = fetch_conesearch(ra, dec, radius=CONE_RADIUS, n=N_MAX)
        elapsed = time.time() - t0

        if df.empty:
            print(f"→ NO DATA  ({elapsed:.1f}s)")
            ddf_alerts[field_name] = df
            # print(f" ddf {field_name} :  try fast search")
            # df_fs = try_fast_search(ra,dec)
            # n_fs = len(df_fs)
            # print(f"ddf {field_name} : fast search result ==>  n = {n_fs}")
            # print(f" ddf {field_name} : try slicing search")
            # df_ss = try_slicing_search(ra,dec)
            # n_ss = len(df_ss)
            # print(f"ddf {field_name} : slicing search result ==> ntot : {n_ss}")
            # ddf_alerts[field_name] = df_ss
            # ty to fetch in time slice
            t0 = time.time()
            df = fetch_conesearch_sliced(
                ra, dec, radius=CONE_RADIUS, n=N_MAX, start=STARTTIME, stop=STOPTIME, step_days=STEPDAYS
            )
            elapsed = time.time() - t0

        # ── Strip the column prefixes for convenience ─────────────────────────
        # Keep original names — stripping here would break downstream
        # column references that rely on the r:/f: prefix.

        # ── Add a field label ─────────────────────────────────────────────────
        df["field"] = field_name

        # ── Ensure boolean dipole flags are proper booleans ───────────────────
        for bool_col in ["r:isDipole", "r:isNegative", "r:dipoleFitAttempted"]:
            if bool_col in df.columns:
                df[bool_col] = (
                    df[bool_col]
                    .map(
                        lambda v: (
                            True
                            if str(v).strip().lower() in ("true", "1", "yes")
                            else (False if str(v).strip().lower() in ("false", "0", "no") else pd.NA)
                        )
                    )
                    .astype("boolean")
                )

        # ── Numeric coercion for flux columns ─────────────────────────────────
        for col in [
            "r:psfFlux",
            "r:psfFluxErr",
            "r:scienceFlux",
            "r:scienceFluxErr",
            "r:templateFlux",
            "r:templateFluxErr",
            "r:apFlux",
            "r:apFluxErr",
            "r:dipoleFluxDiff",
            "r:dipoleMeanFlux",
            "r:dipoleLength",
            "r:dipoleAngle",
            "r:dipoleChi2",
            "r:midpointMjdTai",
            "r:ra",
            "r:dec",
        ]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        df.to_parquet(parquet_path, index=False)
        print(f"→ {len(df):6d} alerts  ({elapsed:.1f}s) → saved in {parquet_path}")

    ddf_alerts[field_name] = df

print("\nFetch complete.")

## 5. Summary statistics per DDF

In [ ]:
rows = []
for field_name, df in ddf_alerts.items():
    if df.empty:
        rows.append(
            {
                "field": field_name,
                "n_alerts": 0,
                "n_dipoles": 0,
                "dipole_fraction": np.nan,
                "n_fit_attempted": 0,
            }
        )
        continue

    n_total = len(df)
    n_dipoles = int(df["r:isDipole"].fillna(False).sum()) if "r:isDipole" in df.columns else 0
    n_fit = int(df["r:dipoleFitAttempted"].fillna(False).sum()) if "r:dipoleFitAttempted" in df.columns else 0
    rows.append(
        {
            "field": field_name,
            "n_alerts": n_total,
            "n_fit_attempted": n_fit,
            "n_dipoles": n_dipoles,
            "dipole_fraction": n_dipoles / n_total if n_total > 0 else np.nan,
            "fit_fraction": n_fit / n_total if n_total > 0 else np.nan,
        }
    )

df_summary = pd.DataFrame(rows).set_index("field")
print("\nDipole summary per DDF:")
print(df_summary.to_string(float_format="{:.4f}".format))

In [ ]:
ddf_alerts["COSMOS"]

In [ ]:
# ── Bar chart: dipole fraction per field ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
fields = df_summary.index.tolist()
fracs = df_summary["dipole_fraction"].fillna(0).values * 100.0
colors = plt.cm.tab10(np.linspace(0, 1, len(fields)))

bars = ax.bar(fields, fracs, color=colors, edgecolor="k", linewidth=0.5)
ax.set_ylabel("Dipole fraction (%)")
ax.set_title("Fraction of DIA alerts flagged as dipoles per DDF")
ax.tick_params(axis="x", rotation=30)
for bar, frac in zip(bars, fracs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f"{frac:.1f}%",
        ha="center",
        va="bottom",
        fontsize=8,
    )
plt.tight_layout()
savefig("dipole_fraction_per_ddf")
plt.show()

## 6. Sky map of dipoles per DDF

For each DDF we plot:
- **grey dots**: all alerts (non-dipole)
- **red arrows**: dipole alerts — arrow direction = `dipoleAngle`, length ∝ `dipoleLength`
- **colour coding** by `r:band`

In [ ]:
def plot_dipole_skymap(df: pd.DataFrame, field_name: str, max_arrows: int = 3000):
    """
    Plot a sky map showing the positions of all alerts and the orientation
    of dipole alerts in (ra, dec) coordinates.

    Parameters
    ----------
    df         : DataFrame with all alerts for one DDF
    field_name : label for the title
    max_arrows : cap on the number of arrows drawn (for performance)
    """
    if df.empty:
        print(f"[{field_name}] No data — skipping sky map.")
        return

    required = {"r:ra", "r:dec", "r:isDipole"}
    missing = required - set(df.columns)
    if missing:
        print(f"[{field_name}] Missing columns {missing} — skipping sky map.")
        return

    df = df.copy()
    df["r:isDipole"] = df["r:isDipole"].fillna(False).astype(bool)

    df_nd = df[~df["r:isDipole"]]  # non-dipole
    df_dp = df[df["r:isDipole"]]  # dipole

    fig, ax = plt.subplots(figsize=(7, 7))

    # --- Background: all non-dipole alerts -----------------------------------
    ax.scatter(
        df_nd["r:ra"].values,
        df_nd["r:dec"].values,
        s=1,
        c="#cccccc",
        alpha=0.3,
        label=f"non-dipole ({len(df_nd):,})",
    )

    # --- Dipole arrows -------------------------------------------------------
    if len(df_dp) > 0 and "r:dipoleAngle" in df_dp.columns and "r:dipoleLength" in df_dp.columns:
        df_plot = df_dp.dropna(subset=["r:ra", "r:dec", "r:dipoleAngle", "r:dipoleLength"])
        if len(df_plot) > max_arrows:
            df_plot = df_plot.sample(max_arrows, random_state=42)
            print(f"[{field_name}] Downsampled arrows to {max_arrows} for display.")

        # dipoleAngle: position angle in degrees (N through E, like PA on sky)
        # Convert to quiver (dx, dy) in (ra, dec) coordinates:
        #   PA=0   → North (+dec)
        #   PA=90  → East  (+ra, but RA increases Westward, so flip sign)
        # Scale: use dipoleLength in arcsec → convert to degrees for plotting
        angle_rad = np.radians(df_plot["r:dipoleAngle"].values)
        length_deg = df_plot["r:dipoleLength"].values / 3600.0  # arcsec → deg
        # Normalise so the arrow length is visible but not dominant
        scale = 0.05 / np.nanmedian(length_deg) if np.nanmedian(length_deg) > 0 else 1.0
        dx = np.sin(angle_rad) * length_deg * scale  # East component  (RA direction, flipped)
        dy = np.cos(angle_rad) * length_deg * scale  # North component (Dec direction)

        # Colour arrows by band if available
        if "r:band" in df_plot.columns:
            for band, grp in df_plot.groupby("r:band"):
                idx = grp.index
                ax.quiver(
                    grp["r:ra"].values,
                    grp["r:dec"].values,
                    dx[df_plot.index.get_indexer(idx)],
                    dy[df_plot.index.get_indexer(idx)],
                    color=BAND_COLORS.get(band, "red"),
                    angles="xy",
                    scale_units="xy",
                    scale=1.0,
                    width=0.002,
                    headwidth=4,
                    headlength=4,
                    label=f"dipole band={band} ({len(grp):,})",
                    alpha=0.7,
                )
        else:
            ax.quiver(
                df_plot["r:ra"].values,
                df_plot["r:dec"].values,
                dx,
                dy,
                color="red",
                angles="xy",
                scale_units="xy",
                scale=1.0,
                width=0.002,
                headwidth=4,
                headlength=4,
                label=f"dipole ({len(df_plot):,})",
                alpha=0.7,
            )
    else:
        ax.scatter(
            df_dp["r:ra"].values,
            df_dp["r:dec"].values,
            s=8,
            c="red",
            alpha=0.5,
            label=f"dipole ({len(df_dp):,})",
        )

    # ---- Axes and labels ----------------------------------------------------
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    ax.set_aspect("auto", "box")
    ax.set_title(
        f"Dipole sky map — {field_name}\n"
        f"total={len(df):,}  dipoles={len(df_dp):,}  "
        f"({100 * len(df_dp) / max(1, len(df)):.1f}%)"
    )
    ax.invert_xaxis()  # standard RA orientation (East to the left)
    ax.legend(loc="best", fontsize=7, markerscale=3)
    plt.tight_layout()
    savefig(f"skymap_dipoles_{field_name.replace('-', '_')}")
    plt.show()


print("plot_dipole_skymap() defined.")

In [ ]:
# ── Plot sky maps for all DDFs ────────────────────────────────────────────────
for field_name, df in ddf_alerts.items():
    try:
        plot_dipole_skymap(df, field_name)
    except Exception as inst:
        print(type(inst))  # the exception type
        print(inst.args)  # arguments stored in .args
        print(inst)  # __str__ allows args to be printed directly,
        # but may be overridden in exception subclasses

## 6b. Dipole direction (angle) distribution per DDF

For each DDF two panels are shown side by side:

- **Left — polar rose diagram**: distribution of `r:dipoleAngle` (position angle of the
  dipole axis, 0° = North, 90° = East, clockwise convention).  
  A **uniform distribution** (dashed red circle) indicates random noise orientation.  
  Any **preferred direction** would betray a systematic shift at the telescope level
  (tracking error, read-out direction, wind shake, …).
- **Right — linear histogram by band**: the same angles split by filter, to check
  whether some bands have a specific preferred orientation.

The `dipoleAngle` column follows the standard astronomical position angle convention.


In [ ]:
def plot_dipole_direction_per_ddf(
    ddf_alerts: dict,
    angle_col: str = "r:dipoleAngle",
    band_col: str = "r:band",
    n_bins_rose: int = 36,
) -> None:
    """
    For each DDF plot two panels side by side:
    Left  — polar rose diagram of dipoleAngle (all dipole alerts combined).
    Right — linear histogram of dipoleAngle split by filter band.

    Parameters
    ----------
    ddf_alerts   : dict  field_name → DataFrame
    angle_col    : column with dipole PA in degrees (0–360)
    band_col     : column with filter band label
    n_bins_rose  : angular bins in the rose diagram (default 36 = 10°/bin)
    """
    fields_with_data = [
        (name, df) for name, df in ddf_alerts.items() if not df.empty and angle_col in df.columns
    ]
    if not fields_with_data:
        print("No dipoleAngle data available.")
        return

    n_fields = len(fields_with_data)
    fig = plt.figure(figsize=(12, 3.8 * n_fields))
    bin_edges_rad = np.linspace(0, 2 * np.pi, n_bins_rose + 1)
    bin_centers = (bin_edges_rad[:-1] + bin_edges_rad[1:]) / 2.0
    bin_width = 2 * np.pi / n_bins_rose

    for row, (field_name, df) in enumerate(fields_with_data):
        # Restrict to dipole-flagged alerts when the flag column is available
        if "r:isDipole" in df.columns:
            df_dip = df[df["r:isDipole"].fillna(False).astype(bool)].copy()
        else:
            df_dip = df.copy()

        angles_deg = pd.to_numeric(df_dip[angle_col], errors="coerce").dropna().values % 360.0
        angles_rad = np.radians(angles_deg)

        # ── Left: polar rose diagram ──────────────────────────────────────────
        ax_pol = fig.add_subplot(n_fields, 2, 2 * row + 1, projection="polar")
        if len(angles_rad) > 0:
            counts_rose, _ = np.histogram(angles_rad, bins=bin_edges_rad)
            freq = counts_rose / counts_rose.sum() if counts_rose.sum() > 0 else counts_rose
            ax_pol.bar(
                bin_centers,
                freq,
                width=bin_width * 0.92,
                bottom=0,
                color="steelblue",
                edgecolor="white",
                linewidth=0.3,
                alpha=0.85,
            )
            # Dashed circle at the uniform level
            uniform_level = 1.0 / n_bins_rose
            ax_pol.plot(
                np.linspace(0, 2 * np.pi, 300),
                np.full(300, uniform_level),
                "--",
                color="crimson",
                lw=1.0,
                alpha=0.8,
                label="uniform",
            )
            ax_pol.legend(loc="lower right", fontsize=6, bbox_to_anchor=(1.25, -0.05))

        ax_pol.set_theta_zero_location("N")  # North at top
        ax_pol.set_theta_direction(-1)  # clockwise (PA convention)
        ax_pol.tick_params(labelsize=7)
        ax_pol.set_title(
            f"{field_name} — rose diagram (n={len(angles_deg):,})",
            va="bottom",
            pad=18,
            fontsize=9,
        )
        # Cardinal labels
        for pa_deg, label in [(0, "N"), (90, "E"), (180, "S"), (270, "W")]:
            ax_pol.text(
                np.radians(pa_deg),
                ax_pol.get_rmax() * 1.2,
                label,
                ha="center",
                va="center",
                fontsize=7,
                fontweight="bold",
            )

        # ── Right: linear histogram by band ──────────────────────────────────
        ax_lin = fig.add_subplot(n_fields, 2, 2 * row + 2)
        lin_edges = np.linspace(0, 360, n_bins_rose + 1)
        if len(angles_deg) == 0:
            ax_lin.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax_lin.transAxes)
        elif band_col in df_dip.columns:
            band_order = [b for b in list("ugrizy") if b in df_dip[band_col].dropna().unique()]
            bottom = np.zeros(n_bins_rose)
            for band in band_order:
                sub = (
                    pd.to_numeric(df_dip.loc[df_dip[band_col] == band, angle_col], errors="coerce")
                    .dropna()
                    .values
                    % 360.0
                )
                if len(sub) == 0:
                    continue
                cnts, _ = np.histogram(sub, bins=lin_edges)
                ax_lin.bar(
                    lin_edges[:-1],
                    cnts,
                    width=360.0 / n_bins_rose * 0.9,
                    bottom=bottom,
                    color=BAND_COLORS.get(band, "grey"),
                    edgecolor="white",
                    linewidth=0.3,
                    label=f"{band} (n={len(sub):,})",
                    alpha=0.85,
                )
                bottom += cnts
        else:
            cnts, _ = np.histogram(angles_deg, bins=lin_edges)
            ax_lin.bar(
                lin_edges[:-1],
                cnts,
                width=360.0 / n_bins_rose * 0.9,
                color="steelblue",
                edgecolor="white",
                linewidth=0.3,
            )

        ax_lin.set_xlabel("Dipole PA (deg, N through E)")
        ax_lin.set_ylabel("N dipole alerts")
        ax_lin.set_xlim(0, 360)
        ax_lin.set_xticks(np.arange(0, 361, 45))
        ax_lin.set_xticklabels(
            ["N\n0°", "45°", "E\n90°", "135°", "S\n180°", "225°", "W\n270°", "315°", "N\n360°"],
            fontsize=7,
        )
        ax_lin.set_title(f"{field_name} — dipoleAngle by band", fontsize=9)
        ax_lin.legend(loc="upper right", fontsize=7, ncol=2)

    fig.suptitle("Dipole orientation per DDF", fontsize=11, y=1.005)
    plt.tight_layout()
    savefig("dipole_direction_per_ddf")
    plt.show()


plot_dipole_direction_per_ddf(ddf_alerts)

## 7. Dipole fraction vs time (per DDF)

We bin alerts by MJD interval (configurable `MJD_BIN_DAYS`) and compute  
for each bin: `n_dipoles / n_total`.  
This allows tracking whether the AP pipeline improved over the observing campaign.

### Utility functions

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """
    Convert an array of MJD (TAI) values to ISO date strings 'YYYY-MM-DD'.

    Uses astropy.time.Time for the conversion.
    """
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 7) -> None:
    """
    Add a secondary x-axis on **top** of *ax* showing calendar dates (YYYY-MM-DD),
    inclined 40° to the left for readability.

    The tick positions share the same data coordinate as the primary MJD axis, so
    zooming/panning the primary axis will keep the dates aligned.

    Parameters
    ----------
    ax         : primary matplotlib Axes (MJD on x)
    mjd_values : array of MJD values present in the current plot
                 (used to determine the range and spacing)
    n_ticks    : target number of date tick marks (capped at the available range)
    """
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return

    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return

    # Evenly-spaced tick positions in MJD coordinates
    n_ticks = max(3, min(n_ticks, len(finite)))
    tick_mjd = np.linspace(mjd_lo, mjd_hi, n_ticks)
    tick_lbls = mjd_to_datestr(tick_mjd)  # list of 'YYYY-MM-DD'

    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())  # must match the primary axis limits exactly
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=7, labelpad=6)


print("mjd_to_datestr() and add_date_axis_on_top() defined.")

In [ ]:
MJD_BIN_DAYS = 1  # bin width in days (change to 1 for nightly resolution)


def compute_dipole_fraction_vs_time(
    df: pd.DataFrame,
    bin_days: float = MJD_BIN_DAYS,
    mjd_col: str = "r:midpointMjdTai",
    flag_col: str = "r:isDipole",
) -> pd.DataFrame:
    """
    Compute the fraction of dipole alerts in time bins.

    Returns a DataFrame with columns:
        mjd_bin_center, n_total, n_dipoles, dipole_fraction,
        dipole_fraction_err  (Poisson: sqrt(n_dipoles) / n_total)
    """
    if df.empty or mjd_col not in df.columns or flag_col not in df.columns:
        return pd.DataFrame()

    df = df.copy()
    df[flag_col] = df[flag_col].fillna(False).astype(bool)
    df[mjd_col] = pd.to_numeric(df[mjd_col], errors="coerce")
    df = df.dropna(subset=[mjd_col])
    if df.empty:
        return pd.DataFrame()

    mjd_min = df[mjd_col].min()
    mjd_max = df[mjd_col].max()
    bins = np.arange(mjd_min, mjd_max + bin_days, bin_days)
    df["mjd_bin"] = pd.cut(df[mjd_col], bins=bins, labels=False)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    rows = []
    for i, center in enumerate(bin_centers):
        mask = df["mjd_bin"] == i
        sub = df[mask]
        n_tot = len(sub)
        n_dip = int(sub[flag_col].sum())
        frac = n_dip / n_tot if n_tot > 0 else np.nan
        frac_e = np.sqrt(n_dip) / n_tot if (n_tot > 0 and n_dip > 0) else np.nan
        rows.append(
            {
                "mjd_bin_center": center,
                "n_total": n_tot,
                "n_dipoles": n_dip,
                "dipole_fraction": frac,
                "dipole_fraction_err": frac_e,
            }
        )

    return pd.DataFrame(rows)


print("compute_dipole_fraction_vs_time() defined.")

In [ ]:
# ── Plot dipole fraction vs time for each DDF ─────────────────────────────────
fig, axes = plt.subplots(
    nrows=len(DEEP_FIELDS),
    ncols=1,
    figsize=(11, 2.5 * len(DEEP_FIELDS)),
    sharex=False,
)
if len(DEEP_FIELDS) == 1:
    axes = [axes]

for ax, (field_name, df) in zip(axes, ddf_alerts.items()):
    df_time = compute_dipole_fraction_vs_time(df, bin_days=MJD_BIN_DAYS)

    if df_time.empty:
        ax.set_title(f"{field_name} — no data")
        continue

    mjd = df_time["mjd_bin_center"].values
    frac = df_time["dipole_fraction"].values * 100.0
    err = df_time["dipole_fraction_err"].fillna(0).values * 100.0

    dt = mjd - mjd.min()

    ax.errorbar(
        mjd,
        frac,
        yerr=err,
        fmt="o-",
        ms=4,
        lw=1.2,
        capsize=3,
        color="steelblue",
        label=f"bin={MJD_BIN_DAYS}d",
    )
    # Twin axis: total number of alerts per bin
    ax2 = ax.twinx()
    ax2.bar(
        mjd,
        df_time["n_total"].values,
        width=MJD_BIN_DAYS * 0.8,
        color="lightgrey",
        alpha=0.4,
        label="N alerts",
    )
    ax2.set_ylabel("N alerts", fontsize=7, color="grey")
    ax2.tick_params(axis="y", labelcolor="grey", labelsize=7)

    ax.set_ylabel("Dipole fraction (%)")
    ax.set_title(f"{field_name} — dipole fraction vs time (bin={MJD_BIN_DAYS} days)")
    ax.set_xlabel("MJD (TAI)")
    ax.legend(loc="upper right", fontsize=7)
    # _add_date_axis(ax, dt, mjd.min())
    add_date_axis_on_top(ax, mjd, n_ticks=7)

plt.tight_layout()
savefig(f"dipole_fraction_vs_time_bin{MJD_BIN_DAYS}d")
plt.show()

## 8. Dipole fraction per visit (per DDF)

Per-visit statistics: fraction of alerts with `isDipole=True` in each visit,  
ordered by visit ID (which encodes the date `YYYYMMDDNNNNN`).
This is a finer-grained view than the time-bin analysis above.

In [ ]:
def compute_dipole_fraction_per_visit(
    df: pd.DataFrame,
    visit_col: str = "r:visit",
    flag_col: str = "r:isDipole",
    mjd_col: str = "r:midpointMjdTai",
    band_col: str = "r:band",
) -> pd.DataFrame:
    """
    Compute the dipole fraction per visit.

    Returns a DataFrame with one row per (visit, band) pair, sorted by visit.
    Columns: visit, band, n_total, n_dipoles, dipole_fraction, mjd_mean.
    """
    if df.empty:
        return pd.DataFrame()
    for col in (visit_col, flag_col):
        if col not in df.columns:
            return pd.DataFrame()

    df = df.copy()
    df[flag_col] = df[flag_col].fillna(False).astype(bool)

    group_cols = [visit_col]
    if band_col in df.columns:
        group_cols.append(band_col)

    agg = (
        df.groupby(group_cols)
        .agg(
            n_total=(flag_col, "count"),
            n_dipoles=(flag_col, "sum"),
            mjd_mean=(mjd_col, "mean") if mjd_col in df.columns else (flag_col, "count"),
        )
        .reset_index()
    )
    agg["dipole_fraction"] = agg["n_dipoles"] / agg["n_total"]
    agg = agg.sort_values(visit_col).reset_index(drop=True)
    return agg


print("compute_dipole_fraction_per_visit() defined.")

In [ ]:
# ── Plot per-visit dipole fraction for each DDF (all bands) ──────────────────
for field_name, df in ddf_alerts.items():
    df_visit = compute_dipole_fraction_per_visit(df)
    if df_visit.empty:
        print(f"[{field_name}] No per-visit data — skipping.")
        continue

    fig, ax = plt.subplots(figsize=(12, 4))

    if "r:band" in df_visit.columns:
        for band, grp in df_visit.groupby("r:band"):
            ax.plot(
                grp["r:visit"].astype(str),
                grp["dipole_fraction"] * 100,
                "o-",
                ms=3,
                lw=1,
                color=BAND_COLORS.get(band, "grey"),
                label=f"band={band}",
            )
    else:
        ax.plot(
            df_visit["r:visit"].astype(str),
            df_visit["dipole_fraction"] * 100,
            "o-",
            ms=3,
            lw=1,
            color="steelblue",
        )

    n_visits = df_visit["r:visit"].nunique()
    # Show only every Nth x-tick to avoid crowding
    tick_step = max(1, n_visits // 20)
    ax.xaxis.set_major_locator(plt.MaxNLocator(20))
    ax.tick_params(axis="x", rotation=45, labelsize=6)

    ax.set_xlabel("Visit ID")
    ax.set_ylabel("Dipole fraction (%)")
    ax.set_title(f"{field_name} — dipole fraction per visit")
    ax.legend(loc="best", fontsize=7)
    plt.tight_layout()
    savefig(f"dipole_fraction_per_visit_{field_name.replace('-', '_')}")
    plt.show()

## 9. Dipole morphology distributions

Distributions of:
- `dipoleLength`  (arcsec) — expected ~PSF size if template mis-registration
- `dipoleAngle`   (degrees) — should be isotropic if random; preferred angles → systematic shifts
- `dipoleChi2`   — goodness of fit
- `dipoleFluxDiff` (nJy)   — flux amplitude of the dipole

for all DDFs combined and per DDF.

In [ ]:
# ── Concatenate all dipole alerts ─────────────────────────────────────────────
frames = []
for field_name, df in ddf_alerts.items():
    if df.empty:
        continue
    if "r:isDipole" not in df.columns:
        continue
    df_dip = df[df["r:isDipole"].fillna(False).astype(bool)].copy()
    df_dip["field"] = field_name
    frames.append(df_dip)

df_all_dipoles = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Total dipole alerts across all DDFs: {len(df_all_dipoles):,}")
if not df_all_dipoles.empty:
    print(df_all_dipoles[["field"]].value_counts().to_string())

In [ ]:
# ── Morphology distributions (combined) ──────────────────────────────────────
if df_all_dipoles.empty:
    print("No dipole data available — skipping morphology plots.")
else:
    morph_cols = {
        "r:dipoleLength": ("Dipole length (arcsec)", 0, 20, 50),
        "r:dipoleAngle": ("Dipole angle (deg)", 0, 360, 36),
        "r:dipoleChi2": ("Dipole chi2", 0, 50, 50),
        "r:dipoleFluxDiff": ("Dipole flux diff (nJy)", None, None, 60),
    }

    fig, axes = plt.subplots(1, len(morph_cols), figsize=(14, 4))

    for ax, (col, (xlabel, xmin, xmax, nbins)) in zip(axes, morph_cols.items()):
        if col not in df_all_dipoles.columns:
            ax.set_title(f"{col}\n(not found)")
            continue
        vals = pd.to_numeric(df_all_dipoles[col], errors="coerce").dropna().values
        if xmin is not None and xmax is not None:
            vals = vals[(vals >= xmin) & (vals <= xmax)]
        ax.hist(vals, bins=nbins, color="steelblue", edgecolor="white", linewidth=0.3)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("N alerts")
        ax.set_title(f"{col.split(':')[1]}\n(n={len(vals):,})")

    plt.suptitle("Dipole morphology — all DDFs combined", y=1.02, fontsize=11)
    plt.tight_layout()
    savefig("dipole_morphology_all_ddfs")
    plt.show()

In [ ]:
# ── dipoleAngle polar histogram (rose diagram) ────────────────────────────────
# A non-isotropic distribution would indicate a preferred direction for
# template mis-registration (e.g. wind shake, tracking error along one axis).
if df_all_dipoles.empty or "r:dipoleAngle" not in df_all_dipoles.columns:
    print("Skipping polar histogram — no dipoleAngle data.")
else:
    angles_deg = pd.to_numeric(df_all_dipoles["r:dipoleAngle"], errors="coerce").dropna().values
    angles_rad = np.radians(angles_deg % 360)

    n_bins = 36
    bin_edges = np.linspace(0, 2 * np.pi, n_bins + 1)
    counts, _ = np.histogram(angles_rad, bins=bin_edges)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2.0
    width = 2 * np.pi / n_bins

    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection="polar")
    ax.bar(
        bin_centers,
        counts,
        width=width * 0.9,
        bottom=0,
        color="steelblue",
        edgecolor="white",
        linewidth=0.4,
        alpha=0.8,
    )
    ax.set_theta_zero_location("N")  # North at top
    ax.set_theta_direction(-1)  # clockwise (astronomical convention)
    ax.set_title(f"Dipole angle distribution\n(all DDFs, n={len(angles_deg):,})", va="bottom", pad=20)
    plt.tight_layout()
    savefig("dipole_angle_polar_histogram")
    plt.show()

## 10. Dipole fraction per band and per DDF

The dipole rate may vary with filter band if the template depth or PSF properties differ.  
We compute a 2D matrix: field × band.

In [ ]:
band_rows = []
for field_name, df in ddf_alerts.items():
    if df.empty or "r:isDipole" not in df.columns or "r:band" not in df.columns:
        continue
    df = df.copy()
    df["r:isDipole"] = df["r:isDipole"].fillna(False).astype(bool)
    for band, grp in df.groupby("r:band"):
        n_tot = len(grp)
        n_dip = int(grp["r:isDipole"].sum())
        band_rows.append(
            {
                "field": field_name,
                "band": band,
                "n_total": n_tot,
                "n_dipoles": n_dip,
                "dipole_fraction": n_dip / n_tot if n_tot > 0 else np.nan,
            }
        )

df_band = pd.DataFrame(band_rows)

if not df_band.empty:
    pivot = df_band.pivot_table(index="field", columns="band", values="dipole_fraction").reindex(
        columns=list("ugrizy")
    )
    print("Dipole fraction per field × band:")
    print(pivot.to_string(float_format="{:.4f}".format))

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(pivot.values * 100, aspect="auto", cmap="YlOrRd", vmin=0, vmax=None)
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns.tolist())
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels(pivot.index.tolist())
    ax.set_xlabel("Band")
    ax.set_ylabel("DDF")
    ax.set_title("Dipole fraction (%) per DDF × band")
    plt.colorbar(im, ax=ax, label="Dipole fraction (%)")
    # Annotate each cell
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val * 100:.1f}", ha="center", va="center", fontsize=8, color="black")
    plt.tight_layout()
    savefig("dipole_fraction_field_vs_band_heatmap")
    plt.show()

## 11. Save concatenated dipole catalogue

Save all dipole alerts from all DDFs into a single parquet file  
for downstream analysis (e.g. cross-matching with Butler visits, PSF fitting, cutout inspection).

In [ ]:
if not df_all_dipoles.empty:
    out_path = os.path.join(DIR_DATA, "all_ddfs_dipoles_only.parquet")
    df_all_dipoles.to_parquet(out_path, index=False)
    print(f"Saved {len(df_all_dipoles):,} dipole alerts → {out_path}")
    print(f"Columns: {list(df_all_dipoles.columns)}")
    display(df_all_dipoles.head(5))
else:
    print("No dipole data — nothing saved.")

## 12. Next steps (future notebooks)

1. **`02_dipole_cutouts.ipynb`** — Fetch image cutouts (`/api/v1/cutouts`) for a  
   representative sample of dipoles per DDF and per band. Visual inspection of the  
   science/template/difference triplets to confirm the dipole morphology.

2. **`03_dipole_sources.ipynb`** — Fetch full diaSource light curves  
   (`/api/v1/sources`) for objects with high `dipole_fraction`, to see whether  
   the same object is repeatedly detected as a dipole (template issue) or only  
   occasionally (transient template artefact).

3. **`04_dipole_butler_cross.ipynb`** — Use `r:visit` + `r:detector` to locate  
   the dipoles in the Butler data store (USDF), load the warp/coadd templates,  
   and characterise the registration quality.

4. **`05_dipole_psf_analysis.ipynb`** — Compare `dipoleLength` with the PSF FWHM  
   per visit to test whether dipoles are consistent with a sub-PSF template shift,  
   and check the correlation with airmass and seeing.